# Omnichannel Retail Analytics & Customer Churn Intelligence
### End-to-End Enterprise Portfolio Project | Aura Retail Group
**Author:** Data Analyst & Analytics Engineer  
**Tech Stack:** Python (pandas, scikit-learn), SQLite/ANSI SQL, Star Schema Dimensional Modeling, Plotly/Seaborn, Power BI

---

## 1. Business Context & Strategic Objectives
Aura Retail Group operates across Web, Mobile App, and Flagship Retail Stores with over 4,500 registered customers and 120 SKUs. While gross revenue reached $3.7M+ across 2024-2025, leadership noted two concerning trends:
1. **Cohort Churn Cliff:** Customer repeat order rates decline precipitously after Month 2.
2. **Margin Erosion:** Aggressive holiday discounting (>20%) fails to drive incremental unit economics, compressing gross margins.

**Objective:** Build an end-to-end analytics platform featuring a Kimball Star Schema warehouse, RFM segmentation, exploratory business visualizations, and a leakage-proof machine learning model that predicts 90-day churn risk and prescribes targeted retention actions.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set path to project root
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, DATABASE_PATH, SQL_DIR
from src.utils.database import WarehouseManager

manager = WarehouseManager(DATABASE_PATH)
print("Environment initialized and database connected successfully!")

## 2. Ingesting Cleaned Data Mart & Dimensional Tables

In [ ]:
dim_customers = pd.read_parquet(PROCESSED_DATA_DIR / "dim_customers.parquet")
dim_products = pd.read_parquet(PROCESSED_DATA_DIR / "dim_products.parquet")
fact_orders = pd.read_parquet(PROCESSED_DATA_DIR / "fact_orders.parquet")

print(f"Active Customers: {len(dim_customers):,}")
print(f"Catalog SKUs:     {len(dim_products):,}")
print(f"Order Lines:      {len(fact_orders):,}")
display(fact_orders.head(3))

## 3. Executive Financial Performance (SQL Warehouse Query)
We execute our production SQL query from `sql/analytics/01_executive_kpis.sql` against the Star Schema database.

In [ ]:
exec_kpis = manager.run_query_file(SQL_DIR / "analytics" / "01_executive_kpis.sql")
display(exec_kpis)

## 4. RFM Customer Behavioral Segmentation
Running window-function based NTILE(5) scoring across Recency, Frequency, and Monetary spend.

In [ ]:
rfm_summary = manager.run_query_file(SQL_DIR / "analytics" / "03_rfm_customer_segmentation.sql")
display(rfm_summary)

# Visualizing Customer Share vs Revenue Share
plt.figure(figsize=(10, 4.5))
y = np.arange(len(rfm_summary))
plt.barh(y - 0.2, rfm_summary["customer_share_pct"], height=0.4, label="% Customers", color="#94A3B8")
plt.barh(y + 0.2, rfm_summary["revenue_share_pct"], height=0.4, label="% Revenue", color="#0284C7")
plt.yticks(y, rfm_summary["customer_segment"])
plt.gca().invert_yaxis()
plt.xlabel("Percentage (%)")
plt.title("RFM Behavioral Segments: Customer Concentration vs Revenue Share")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Machine Learning Churn Benchmark & Financial ROI Curve
Evaluating the trained HistGradientBoosting pipeline on the holdout test set with zero-leakage observation cutoffs.

In [ ]:
pipeline = joblib.load(PROJECT_ROOT / "models" / "churn_pipeline.joblib")
scored_customers = pd.read_parquet(PROCESSED_DATA_DIR / "scored_customers_retention.parquet")

print("Trained Pipeline Architecture:")
print(pipeline)

print("\nScored Risk Distribution:")
print(scored_customers["risk_tier"].value_counts())
display(scored_customers[["customer_id", "recency_days", "frequency", "total_net_spend", "churn_probability", "risk_tier", "prescribed_action"]].head(5))

## 6. Strategic Business Recommendations
1. **Eliminate Clearance Discounts >30%:** Discounts above 20% erode gross margin to 15.6% while increasing return rates to 11.2%.
2. **Implement Day-45 Automated Re-engagement:** Target customers before the 90-day churn cliff, generating an estimated **$100,020 in annual net saved margin**.
3. **Prioritize Mobile App Channel:** Mobile app customers have a 24% higher repeat order rate and $42 higher AOV than desktop web shoppers.